# Exploring E3SM FATES ELM NetCDF Output — a plotting cookbook

This notebook is a systematic tour of how to plot **every distinct dimension
signature** found in an E3SM/FATES land-model history file. Rather than
picking a few favorite variables, it groups all ~630 variables by their
dimension combination (e.g. `(time, fates_levpft, lndgrid)`), and shows the
plot type that actually fits each shape:

- **1D static arrays** (index/mapping tables, vertical profiles) → bar / stem / profile plots
- **`(time, lndgrid)`** (the most common shape) → simple time series line plots
- **`(time, category, lndgrid)`** → heatmaps (category × time), multi-line plots, or stacked-area plots depending on category count and meaning
- **flattened cross-product categories** (e.g. land-use × land-use transition matrices) → true 2D **matrix plots**, reshaped and annotated

File: `E3SM_FATES_TEST.elm.h0.2001-01.nc` (aggregated: `Aggregated_E3SM_FATES_TEST_Output.nc`)

In [1]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import nc_time_axis  # registers a matplotlib unit converter for cftime datetime objects
from collections import defaultdict

plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.autolayout'] = True

ds = xr.open_dataset('../archive/E3SM_FATES_TEST/lnd/hist/Aggregated_E3SM_FATES_TEST_Output.nc')
ds

<xarray.Dataset> Size: 152kB
Dimensions:                               (fates_levscag: 91,
                                           fates_levscpf: 182,
                                           fates_levcapf: 28,
                                           fates_levcnlf: 90,
                                           fates_levcnlfpf: 1260,
                                           fates_levscagpf: 1274,
                                           ...
                                           levsoi: 10, fates_levfuel: 6,
                                           fates_levcacls: 2,
                                           fates_levlupft: 70, levdcmp: 15,
                                           ltype: 9, natpft: 19, levsno: 5,
                                           fates_levcwdsc: 4,
                                           fates_levleaf: 30, fates_levcdam: 2)
Coordinates: (12/17)
  * time                                  (time) object 96B 2001-02-01 00:00:...
  * levgrnd                               (levgrnd) float32 60B 0.007101 ... ...
  * levlak                                (levlak) float32 40B 0.05 ... 44.78
  * fates_levscls                         (fates_levscls) float32 52B 0.0 ......
  * fates_levlanduse                      (fates_levlanduse) int32 20B 1 2 3 4 5
  * fates_levage                          (fates_levage) float32 28B 0.0 ... ...
    ...                                    ...
  * fates_levfuel                         (fates_levfuel) int32 24B 1 2 3 4 5 6
  * fates_levcacls                        (fates_levcacls) float32 8B 0.0 5.0
  * levdcmp                               (levdcmp) float32 60B 0.007101 ... ...
  * fates_levcwdsc                        (fates_levcwdsc) int32 16B 1 2 3 4
  * fates_levleaf                         (fates_levleaf) float32 120B 0.0 .....
  * fates_levcdam                         (fates_levcdam) float32 8B 0.0 80.0
Dimensions without coordinates: fates_levscag, fates_levscpf, fates_levcapf,
                                fates_levcnlf, fates_levcnlfpf,
                                fates_levscagpf, fates_levagepft,
                                fates_levelpft, fates_levelcwd, fates_levelage,
                                fates_levagefuel, fates_levcdsc, fates_levcdpf,
                                hist_interval, lndgrid, fates_levlulu,
                                fates_levlupft, ltype, natpft, levsno
Data variables: (12/630)
    fates_scmap_levscag                   (fates_levscag) int32 364B ...
    fates_agmap_levscag                   (fates_levscag) int32 364B ...
    fates_pftmap_levscpf                  (fates_levscpf) int32 728B ...
    fates_scmap_levscpf                   (fates_levscpf) int32 728B ...
    fates_pftmap_levcapf                  (fates_levcapf) int32 112B ...
    fates_camap_levcapf                   (fates_levcapf) int32 112B ...
    ...                                    ...
    WA                                    (time, lndgrid) float32 48B ...
    WASTEHEAT                             (time, lndgrid) float32 48B ...
    WIND                                  (time, lndgrid) float32 48B ...
    ZBOT                                  (time, lndgrid) float32 48B ...
    ZWT                                   (time, lndgrid) float32 48B ...
    ZWT_PERCH                             (time, lndgrid) float32 48B ...
Attributes: (12/28)
    title:                                     ELM History file information
    source:                                    E3SM Land Model
    source_id:                                 unknown
    product:                                   model-output
    realm:                                     land
    case:                                      E3SM_FATES_TEST
    ...                                        ...
    ltype_deep_lake:                           5
    ltype_wetland:                             6
    ltype_urban_tbd:                           7
    ltype_urban_hd:                    

## 1. Dataset Overview: Dimensions

In [ ]:
print("Dimensions:")
for dim, size in ds.sizes.items():
    print(f"  {dim}: {size}")

## 2. Every Dimension Signature Present in the File

Instead of listing all 630 variables, group them by their exact tuple of dimensions. Each group below gets its own plotting example in Section 4.

In [ ]:
dim_to_vars = defaultdict(list)
for name, var in ds.variables.items():
    if name in ds.coords:
        continue
    dim_to_vars[var.dims].append(name)

print(f"{len(dim_to_vars)} distinct dimension signatures, {sum(len(v) for v in dim_to_vars.values())} variables total\n")
for dims, names in sorted(dim_to_vars.items(), key=lambda kv: (len(kv[0]), kv[0])):
    print(f"{dims!s:70s} {len(names):4d} vars   e.g. {names[:3]}")

## 3. Plotting Toolkit

Small set of reusable helpers, one per plot archetype. Every FATES/ELM
variable in this file falls into one of these archetypes based on its
dimensions:

| Archetype | Dimension shape | Function |
|---|---|---|
| Index/mapping table | `(fates_lev*,)` only, no time | `plot_index_map` |
| Static gridcell scalar | `(lndgrid,)` only | `plot_static_scalars` |
| Static vertical profile | `(depth_dim, lndgrid)` | `plot_profile` |
| Metadata time series | `(time,)` | `plot_timeseries` |
| Simple time series | `(time, lndgrid)` | `plot_timeseries` |
| Category × time (many categories) | `(time, cat, lndgrid)` | `plot_heatmap` |
| Category × time (few categories) | `(time, cat, lndgrid)` | `plot_lines_by_category` |
| Compositional fractions | `(time, cat, lndgrid)` | `plot_stacked_area` |
| Interval / bounds | `(time, hist_interval)` | `plot_intervals` |
| Cross-product matrix | `(time, cat1*cat2, lndgrid)` | `plot_matrix_snapshot` |

All helpers drop the trivial `lndgrid` dimension (size 1 for this single-point
test case) and handle all-NaN slices gracefully (e.g. lake/snow variables at
a snow-free tropical gridcell) instead of raising an error.

In [ ]:
def _sel1(da):
    """Drop the trivial size-1 lndgrid dimension."""
    return da.isel(lndgrid=0) if 'lndgrid' in da.dims else da


def _is_all_nan(arr):
    arr = np.asarray(arr, dtype='float64')
    return not np.isfinite(arr).any()


def _no_data_panel(ax, varname, dims):
    ax.text(0.5, 0.5, "all values are NaN at this gridcell\n(e.g. no lake / no snow present)",
            ha='center', va='center', transform=ax.transAxes, fontsize=9, color='gray')
    ax.set_title(f"{varname}  dims={dims}")
    ax.set_xticks([]); ax.set_yticks([])


def plot_index_map(varname, ax=None):
    """1D static index/mapping array, e.g. fates_scmap_levscag."""
    da = ds[varname]
    ax = ax or plt.gca()
    ax.bar(np.arange(da.size), da.values, color='steelblue')
    ax.set_title(varname, fontsize=9)
    ax.set_xlabel(da.dims[0], fontsize=8)
    ax.tick_params(labelsize=7)
    return ax


def plot_static_scalars(varnames, ax=None):
    """Table of dims=(lndgrid,) static scalars (heterogeneous units)."""
    ax = ax or plt.gca()
    ax.axis('off')
    rows = [[v, f"{float(_sel1(ds[v]).values):.4g}", ds[v].attrs.get('units', '')] for v in varnames]
    table = ax.table(cellText=rows, colLabels=['variable', 'value', 'units'],
                      cellLoc='left', colLoc='left', loc='center')
    table.auto_set_font_size(False); table.set_fontsize(9); table.scale(1, 1.6)
    ax.set_title('Static gridcell-scalar variables  dims=(lndgrid,)')
    return ax


def plot_profile(varname, depth_dim, ax=None):
    """Static vertical profile: variable vs. depth at the single gridcell."""
    da = _sel1(ds[varname])
    ax = ax or plt.gca()
    if _is_all_nan(da.values):
        return _no_data_panel(ax, varname, ds[varname].dims)
    depth = ds[depth_dim].values
    ax.plot(da.values, depth, marker='o')
    ax.invert_yaxis()
    ax.set_xlabel(ds[varname].attrs.get('units', ''))
    ax.set_ylabel(f'{depth_dim} (depth)')
    ax.set_title(f'{varname}  dims={ds[varname].dims}')
    return ax


def plot_timeseries(varname, ax=None):
    """(time,) or (time, lndgrid) line plot."""
    da = _sel1(ds[varname])
    ax = ax or plt.gca()
    ax.plot(ds['time'].values, np.asarray(da.values, dtype='float64'), marker='o')
    ax.set_title(f'{varname}  dims={ds[varname].dims}', fontsize=10)
    ax.set_ylabel(ds[varname].attrs.get('units', ''))
    ax.tick_params(axis='x', rotation=45)
    return ax


def plot_heatmap(varname, cat_dim, ax=None, cmap='viridis'):
    """(time, cat_dim, lndgrid) -> heatmap of category (y) x time (x)."""
    da = _sel1(ds[varname]).transpose(cat_dim, 'time')
    ax = ax or plt.gca()
    data = np.asarray(da.values, dtype='float64')
    if _is_all_nan(data):
        return _no_data_panel(ax, varname, ds[varname].dims)
    im = ax.pcolormesh(np.arange(data.shape[1] + 1), np.arange(data.shape[0] + 1),
                        data, cmap=cmap, shading='flat')
    ax.set_xticks(np.arange(data.shape[1]) + 0.5)
    ax.set_xticklabels([str(t)[:7] for t in ds['time'].values], rotation=45, ha='right', fontsize=7)
    ax.set_yticks(np.arange(data.shape[0]) + 0.5)
    ax.set_yticklabels(np.arange(data.shape[0]))
    ax.set_ylabel(cat_dim)
    plt.colorbar(im, ax=ax, label=ds[varname].attrs.get('units', ''))
    ax.set_title(f'{varname}  dims={ds[varname].dims}', fontsize=10)
    return ax


def plot_lines_by_category(varname, cat_dim, ax=None):
    """(time, cat_dim, lndgrid) -> one line per category, for small category counts."""
    da = _sel1(ds[varname])
    ax = ax or plt.gca()
    for i in range(da.sizes[cat_dim]):
        ax.plot(ds['time'].values, np.asarray(da.isel({cat_dim: i}).values, dtype='float64'),
                 marker='o', label=f'{cat_dim}={i}')
    ax.legend(fontsize=7, ncol=2)
    ax.set_title(f'{varname}  dims={ds[varname].dims}', fontsize=10)
    ax.tick_params(axis='x', rotation=45)
    return ax


def plot_stacked_area(varname, cat_dim, ax=None):
    """(time, cat_dim, lndgrid) -> stacked-area plot, for compositional/fractional variables."""
    da = _sel1(ds[varname])
    ax = ax or plt.gca()
    series = [np.asarray(da.isel({cat_dim: i}).values, dtype='float64') for i in range(da.sizes[cat_dim])]
    labels = [f'{cat_dim}={i}' for i in range(da.sizes[cat_dim])]
    ax.stackplot(ds['time'].values, *series, labels=labels)
    ax.legend(fontsize=6, ncol=2, loc='upper left', bbox_to_anchor=(1.01, 1))
    ax.set_title(f'{varname}  dims={ds[varname].dims}', fontsize=10)
    ax.tick_params(axis='x', rotation=45)
    return ax


def plot_intervals(varname='time_bounds', ax=None):
    """(time, hist_interval) -> Gantt-style bar showing each averaging interval."""
    da = ds[varname]
    ax = ax or plt.gca()
    starts = da.isel(hist_interval=0).values
    ends = da.isel(hist_interval=1).values
    for i, (s, e) in enumerate(zip(starts, ends)):
        ax.barh(i, (e - s).days, left=0, height=0.6, color='seagreen')
        ax.text(0.02, i, f'{s} -> {e}', va='center', fontsize=7)
    ax.set_yticks(range(len(starts))); ax.set_yticklabels([f't={i}' for i in range(len(starts))])
    ax.set_xlabel('interval length (days)')
    ax.set_title(f'{varname}  dims={da.dims}')
    return ax


def plot_matrix_snapshot(varname, row_dim_size, col_dim_size, time_idx=-1, ax=None, cmap='magma'):
    """Reshape a flattened (row*col) categorical dim into a true 2D matrix at one timestep,
    e.g. land-use x land-use transition matrices, or land-use x PFT matrices."""
    da = _sel1(ds[varname]).isel(time=time_idx)
    mat = np.asarray(da.values, dtype='float64').reshape(row_dim_size, col_dim_size)
    ax = ax or plt.gca()
    im = ax.imshow(mat, cmap=cmap, aspect='auto')
    vmax = np.nanmax(mat) if np.isfinite(mat).any() else 0
    for (i, j), v in np.ndenumerate(mat):
        ax.text(j, i, f'{v:.2g}', ha='center', va='center', fontsize=6,
                color='white' if vmax and v > vmax / 2 else 'black')
    ax.set_xlabel('column category index'); ax.set_ylabel('row category index')
    plt.colorbar(im, ax=ax, label=ds[varname].attrs.get('units', ''))
    t = str(ds['time'].values[time_idx])[:7]
    ax.set_title(f'{varname} @ {t}  (reshaped {row_dim_size}x{col_dim_size})', fontsize=10)
    return ax

## 4. One Example Plot per Dimension Signature

### 4.1 Static index/mapping arrays — `(fates_lev*,)` only, no time

FATES stores a family of small internal lookup tables that map one indexing
scheme onto another (e.g. "which PFT does size-class-by-PFT slot *i* belong
to"). These are all 1D, static, with no time dimension. There are 13 such
dimension signatures in the file — shown here as small multiples, one bar
plot per signature (first variable of each group).

In [ ]:
index_map_groups = {dims: names for dims, names in dim_to_vars.items()
                     if len(dims) == 1 and dims[0] != 'time'}

n = len(index_map_groups)
ncols = 4
nrows = -(-n // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 2.5 * nrows))
axes = np.atleast_1d(axes).ravel()
for ax, (dims, names) in zip(axes, sorted(index_map_groups.items())):
    plot_index_map(names[0], ax=ax)
for ax in axes[n:]:
    ax.axis('off')
fig.suptitle('One representative variable per static index/mapping dimension signature', y=1.02)
plt.show()

### 4.2 Time-only metadata — `(time,)`

Numeric run-bookkeeping variables (`nstep`, `mcdate`, `mdcur`, `mscur`, `mcsec`). `date_written`/`time_written` are text strings, printed rather than plotted.

In [ ]:
time_only_numeric = [v for v in dim_to_vars[('time',)] if np.issubdtype(ds[v].dtype, np.number)]
time_only_text = [v for v in dim_to_vars[('time',)] if v not in time_only_numeric]

fig, axes = plt.subplots(1, len(time_only_numeric), figsize=(4 * len(time_only_numeric), 3))
for ax, v in zip(axes, time_only_numeric):
    plot_timeseries(v, ax=ax)
plt.show()

for v in time_only_text:
    print(v, '->', ds[v].values)

### 4.3 Static gridcell scalars — `(lndgrid,)`

Single value per gridcell (lon, lat, area, landfrac, ...). Units differ too much for a shared bar chart to be meaningful, so these are shown as a table.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
plot_static_scalars(dim_to_vars[('lndgrid',)], ax=ax)
plt.show()

### 4.4 Averaging intervals — `(time, hist_interval)`

`time_bounds` records the start/end of each monthly averaging window — a natural fit for a Gantt-style interval plot rather than a line or heatmap.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
plot_intervals('time_bounds', ax=ax)
plt.show()

### 4.5 Static vertical soil profile — `(levgrnd, lndgrid)`

`WATSAT` (saturated soil water content) varies by soil layer but not time — a classic depth profile plot with depth on the inverted y-axis.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 5))
plot_profile('WATSAT', 'levgrnd', ax=ax)
plt.show()

### 4.6 Static lake-layer profile — `(levlak, lndgrid)`

`DZLAKE` (lake layer thickness). This test gridcell has zero lake fraction, so the variable is entirely fill-valued (NaN) — the helper detects this and reports it instead of raising an error or silently drawing an empty axes.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
plot_profile('DZLAKE', 'levlak', ax=ax)
plt.show()

### 4.7 Simple time series — `(time, lndgrid)`

The single most common dimension signature (427 variables). `FATES_GPP` (gross primary production) shown as a standard line plot.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
plot_timeseries('FATES_GPP', ax=ax)
plt.show()

### 4.8 Size-class dynamics — `(time, fates_levscls, lndgrid)` — heatmap

13 tree-size classes evolving over 12 months. Too many categories for a readable legend, so this is a natural heatmap: size class on y, time on x, color = value.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_heatmap('FATES_NPLANT_CANOPY_SZ', 'fates_levscls', ax=ax)
plt.show()

### 4.9 Land-use composition — `(time, fates_levlanduse, lndgrid)` — stacked area

`FATES_PATCHAREA_LU` is a fractional composition (patch area by land-use type, summing to ~1) — a stacked-area plot communicates the composition-over-time story better than a heatmap.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_stacked_area('FATES_PATCHAREA_LU', 'fates_levlanduse', ax=ax)
plt.show()

### 4.10 Patch-age dynamics — `(time, fates_levage, lndgrid)` — heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_heatmap('FATES_LAI_AP', 'fates_levage', ax=ax)
plt.show()

### 4.11 Canopy-height-bin dynamics — `(time, fates_levheight, lndgrid)` — heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_heatmap('FATES_CANOPYAREA_HT', 'fates_levheight', ax=ax)
plt.show()

### 4.12 Per-PFT dynamics — `(time, fates_levpft, lndgrid)` — multi-line

14 PFTs is still small enough for a legend to stay readable, so PFT-level variables are shown as one line per PFT rather than a heatmap — useful when you want to compare individual trajectories rather than scan for overall structure.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_lines_by_category('FATES_GPP_PF', 'fates_levpft', ax=ax)
plt.show()

### 4.13 Per-canopy-layer dynamics — `(time, fates_levcan, lndgrid)` — multi-line

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
plot_lines_by_category('FATES_CROWNAREA_CL', 'fates_levcan', ax=ax)
plt.show()

### 4.14 Land-use transition matrix — `(time, fates_levlulu, lndgrid)` — true 2D matrix plot

`fates_levlulu` (25 = 5x5) is a *flattened cross-product* of (from land-use type) x (to land-use type). Reshaping it back to 5x5 at a single timestep gives a genuine transition matrix — the clearest case for `imshow`-style matrix plotting rather than a 1D heatmap.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
plot_matrix_snapshot('FATES_DISTURBANCE_RATE_MATRIX_LULU', row_dim_size=5, col_dim_size=5, time_idx=-1, ax=ax)
plt.show()

### 4.15 Element (nutrient) flux variables — `(time, fates_levelem, lndgrid)`

`fates_levelem` has size 1 in this run (carbon only tracked), so per-element faceting collapses to a plain time series; more informative here is overlaying several `_EL` flux variables to compare litter fluxes directly.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for v in ['FATES_LITTER_IN_EL', 'FATES_LITTER_OUT_EL']:
    da = _sel1(ds[v]).isel(fates_levelem=0)
    ax.plot(ds['time'].values, np.asarray(da.values, dtype='float64'), marker='o', label=v)
ax.legend(fontsize=8)
ax.set_title('Litter element fluxes  dims=(time, fates_levelem, lndgrid)')
ax.tick_params(axis='x', rotation=45)
plt.show()

### 4.16 Soil-layer (biogeochemistry) dynamics — `(time, levsoi, lndgrid)` — heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_heatmap('FATES_FROOTC_SL', 'levsoi', ax=ax)
plt.show()

### 4.17 Fuel-class dynamics — `(time, fates_levfuel, lndgrid)` — heatmap

This test run had no fire activity, so `FATES_FUEL_AMOUNT_FC` is structurally all zero — the plot still renders correctly (a flat-colored heatmap), demonstrating the method works even on a degenerate/inactive process.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_heatmap('FATES_FUEL_AMOUNT_FC', 'fates_levfuel', ax=ax)
plt.show()

### 4.18 Coarse-woody-debris size-class dynamics — `(time, fates_levelcwd, lndgrid)` — heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
plot_heatmap('FATES_LITTER_CWD_ELDC', 'fates_levelcwd', ax=ax)
plt.show()

### 4.19 Cohort-age-class dynamics — `(time, fates_levcacls, lndgrid)` — multi-line

Only 2 age classes, both structurally zero in this short test run — still shown to confirm the plotting path handles a minimal/degenerate category count.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
plot_lines_by_category('FATES_NPLANT_AC', 'fates_levcacls', ax=ax)
plt.show()

### 4.20 Land-use x PFT matrix — `(time, fates_levlupft, lndgrid)` — true 2D matrix plot

`fates_levlupft` (70 = 5 land-use types x 14 PFTs) is another flattened cross-product — reshaped to a 5x14 matrix at one timestep, same technique as the land-use transition matrix.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
plot_matrix_snapshot('FATES_VEGC_LUPF', row_dim_size=5, col_dim_size=14, time_idx=-1, ax=ax)
plt.show()

### 4.21 Decomposition-layer dynamics — `(time, levdcmp, lndgrid)` — heatmap

46 variables share this signature — `HR_vr` (heterotrophic respiration by depth) shown as a depth x time heatmap, the classic view for a soil profile evolving through the year.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_heatmap('HR_vr', 'levdcmp', ax=ax)
plt.show()

### 4.22 Soil temperature profile — `(time, levgrnd, lndgrid)` — heatmap

`TSOI` by depth and month — a heatmap here reveals the seasonal thermal wave propagating down the soil column, which a set of 15 overlaid lines would not show nearly as clearly.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_heatmap('TSOI', 'levgrnd', ax=ax, cmap='inferno')
plt.show()

### 4.23 Lake-layer dynamics — `(time, levlak, lndgrid)` — heatmap (no data at this site)

`TLAKE`: all-NaN at this gridcell (no lake fraction), same as the static lake profile in 4.6 — included to show the heatmap helper degrades gracefully rather than erroring.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
plot_heatmap('TLAKE', 'levlak', ax=ax)
plt.show()

### 4.24 Land-unit-type composition — `(time, ltype, lndgrid)` — stacked area

`PCT_LANDUNIT` is a percentage composition across 9 land-unit types (vegetated, glacier, lake, wetland, urban, ...) summing to 100 — stacked area again fits this compositional shape.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_stacked_area('PCT_LANDUNIT', 'ltype', ax=ax)
plt.show()

### 4.25 Natural-PFT composition — `(time, natpft, lndgrid)` — heatmap

`PCT_NAT_PFT` spans 19 natural PFT categories — too many for a readable stacked-area legend, so a heatmap is used instead, same reasoning as the size-class case in 4.8.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_heatmap('PCT_NAT_PFT', 'natpft', ax=ax)
plt.show()

### 4.26 Snow-layer dynamics — `(time, levsno, lndgrid)` — heatmap (no data at this site)

`QSNOMELT_LYR`: all-NaN, since this is a snow-free tropical (Brazil) test gridcell — another example of the graceful no-data path rather than a crash.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
plot_heatmap('QSNOMELT_LYR', 'levsno', ax=ax)
plt.show()

## 5. Cheat Sheet: Choosing a Plot Type from Dimensions

| Dimensions after dropping `lndgrid` | What it is | Plot type |
|---|---|---|
| `()` (0-D, per gridcell) | Single static value | table / text |
| `(cat,)`, no time | Static index/mapping table | bar / stem |
| `(depth,)`, no time | Static vertical profile | line vs. depth (inverted y) |
| `(time,)` | Time series (scalar or bookkeeping) | line plot |
| `(time, cat)`, few categories (<~6) | Category trajectories | multi-line with legend |
| `(time, cat)`, many categories | Category trajectories | heatmap (cat x time) |
| `(time, cat)`, fractional / sums to ~1 or 100 | Composition over time | stacked area |
| `(time, depth)` | Profile evolving in time | heatmap (depth x time) |
| `(time, hist_interval)` | Averaging-window bounds | interval / Gantt bar |
| `(time, cat1*cat2)` flattened cross-product | Transition/allocation matrix | reshape → 2D matrix (`imshow`), one snapshot in time |

General rule: **the more categories, the more a heatmap or matrix beats a
legend** — legends stop being readable past roughly 6-8 series, at which
point color-encoded 2D plots communicate structure far better than
overlapping lines.